In [1]:
import main
import pandas as pd
import numpy as np

SapModel = main.main() # always select 0 when using with notebook

import ETABSv1 as etabs
from csiapi import csiutils

def all_column_design_forces(sapmodel): #this will pull all the data from db, quite faster than reading line by line

    result = sapmodel.DatabaseTables.GetTableForDisplayArray(
        "Design Forces - Columns",
        [str()],
        str(),
        int(),
        [str()],
        int(),
        [str()]
    )

    ret = result[0]

    if ret != 0:
        raise RuntimeError("Could not retrieve column design forces table")

    # table_version = result[1]
    field_keys = list(result[1])
    # group_name = result[3]
    # num_records = result[4]
    table_data = list(result[5])

    if len(field_keys)>1:
        n_fields = len(field_keys)
    else:
        print("No field key identified, user defined headers will be used")
        field_keys = ["Story","Column","Unique_Name","Combo","Station","P","V2","V3","T","M2","M3"]
        n_fields = len(field_keys)

    # reshape 1D list to 2D
    data_array = np.array(table_data).reshape(-1, n_fields)

    df = pd.DataFrame(data_array, columns=field_keys)
    if "Frame" in df.columns:
        df.rename(columns={"Frame": "Unique_Name"}, inplace=True)

    numeric_cols = ["P", "V2", "V3", "T", "M2", "M3"]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col])

    return df

Attached file is M9856-M08-ULS-R00-t06.EDB


In [39]:
from csiapi import csiutils,ops,utils

if SapModel.DesignConcrete.GetResultsAvailable(): #if results available we will no design again
    design_concrete = ops.DesignConcrete(SapModel)
    df = design_concrete.col_concdesign_forces("213")
else:
    from csiapi import ops
    design_concrete = ops.DesignConcrete(SapModel)
    df = design_concrete.all_column_design_forces()


Warning!!!! Showing results only for members with available design results


In [37]:
if SapModel.DesignConcrete.GetResultsAvailable(): #if results available we will no design again
    # etabs.cDesignConcrete(SapModel.DesignConcrete)
    df = all_column_design_forces(SapModel)
else:
    from csiapi import ops
    design_concrete = ops.DesignConcrete(SapModel)
    df = design_concrete.all_column_design_forces()

No field key identified, user defined headers will be used


In [ ]:
# delta_ns calculation
import numpy as np

memb_UN = "213"
b_dns = 0.8

section_name = csiutils.get_section(SapModel,memb_UN)
material_name = csiutils.get_prop_material(SapModel,section_name)
fc_ = csiutils.get_matprop(SapModel,material_name)[0]/1000 #MPa
[cover,circle_bars,bars_along_3,bars_along_2, bar_dia] = csiutils.get_colrebar(SapModel,section_name)[5:10]

prop_df = csiutils.get_frameprop(SapModel)
t2 = prop_df[prop_df.Frame_Name == section_name].width.item() * 1000 #mm
t3 = prop_df[prop_df.Frame_Name == section_name].depth.item() * 1000 #mm
b = t2 #assuming 2 axis along width
h = t3 #assuming  3 axis along depth
cover = cover*1000 #mm
bar_dia = int(bar_dia)

As = np.pi * bar_dia**2 / 4


y_minor = b/2 - cover - bar_dia/2
Is_minor = 2 * bars_along_2 * As * y_minor**2

y_major = h/2 - cover - bar_dia/2
Is_major = 2 * bars_along_3 * As * y_major**2

df["Ec"] = 4700 * np.sqrt(fc_)   # MPa
Es = 200000      

df_target = df[df["FrameName"] == memb_UN].copy() #Use .copy() to avoid SettingWithCopyWarning


df_target["Ig2"]  = h*b**3/12   # bending about local 2 axis
df_target["Ig3"]  = b*h**3/12   # bending about local 3 axis


# function to get ratio of moments
group_cols = ["FrameName", "ComboName"]


def get_ratio(g, moment_col):
    first = g.loc[g["Station"].idxmin(), moment_col]
    last  = g.loc[g["Station"].idxmax(), moment_col]

    return first/last if abs(first) < abs(last) else last/first

ratio_m2 = (
    df_target.groupby(group_cols)[["Station","M2"]]
      .apply(lambda g: get_ratio(g, "M2"))
      .rename("M2_ratio")
)

ratio_m3 = (
    df_target.groupby(group_cols)[["Station","M3"]]
      .apply(lambda g: get_ratio(g, "M3"))
      .rename("M3_ratio")
)

df_target = df_target.merge(ratio_m2, on=group_cols)
df_target = df_target.merge(ratio_m3, on=group_cols)

df_target["Cm2"] = np.maximum(0.4, 0.6 + 0.4*df_target["M2_ratio"])
df_target["Cm3"] = np.maximum(0.4, 0.6 + 0.4*df_target["M3_ratio"])
df_target["Lu"] = df_target.groupby(["ComboName"])["Station"].transform("max")
df_target["Lu"] = pd.to_numeric(df_target["Lu"], errors='coerce') # Converts 'N/A' to NaN
df_target["Lu"] *= 1000


df_target["EIeff2"] = (0.2*df_target["Ec"]*df_target["Ig2"] + Es*Is_minor)/(1+b_dns)
df_target["EIeff3"] = (0.2*df_target["Ec"]*df_target["Ig3"] + Es*Is_major)/(1+b_dns)


df_target["Pcr2"] = (np.pi**2 * df_target["EIeff2"]) / (df_target["Lu"]**2)/1000
df_target["Pcr3"] = (np.pi**2 * df_target["EIeff3"]) / (df_target["Lu"]**2)/1000


# df_target = df_target.drop(columns=['V2',"V3"])
df_target["delta_ns_2"] = df_target["Cm2"] / (1 - df_target["P"].abs()/(0.75*df_target["Pcr2"]))
df_target["delta_ns_3"] = df_target["Cm3"] / (1 - df_target["P"].abs()/(0.75*df_target["Pcr3"]))


df_target.sort_values("delta_ns_2",ascending=False).head(20)

,FrameName,ComboName,Station,P,V2,V3,T,M2,M3,Ec,...,M3_ratio,Cm2,Cm3,Lu,EIeff2,EIeff3,Pcr2,Pcr3,delta_ns_2,delta_ns_3
13244,213,UQ02_2_2 - 0.9DL+1.2FL-EQx +0.3EQy (STAGED) - 5,0.000000,-2236.378299,36.234022,-2.592682,-0.850161,10.153594,147.599365,36406.043454,...,-0.130149,0.986393,0.547941,4400.0,2.050819e+13,1.666616e+14,10454.945354,84963.021227,1.379973,0.567870
13258,213,UQ02_2_2 - 0.9DL+1.2FL-EQx +0.3EQy (STAGED) - 7,0.000000,-2236.378299,36.234022,-3.317251,-0.850161,10.153594,59.092503,36406.043454,...,0.333451,0.986393,0.733380,4400.0,2.050819e+13,1.666616e+14,10454.945354,84963.021227,1.379973,0.760055
13246,213,UQ02_2_2 - 0.9DL+1.2FL-EQx +0.3EQy (STAGED) - 5,1.466667,-2227.624939,36.234022,-2.592682,-0.850161,11.054547,92.687264,36406.043454,...,-0.130149,0.986393,0.547941,4400.0,2.050819e+13,1.666616e+14,10454.945354,84963.021227,1.377822,0.567790
13260,213,UQ02_2_2 - 0.9DL+1.2FL-EQx +0.3EQy (STAGED) - 7,1.466667,-2227.624939,36.234022,-3.317251,-0.850161,11.054547,46.317341,36406.043454,...,0.333451,0.986393,0.733380,4400.0,2.050819e+13,1.666616e+14,10454.945354,84963.021227,1.377822,0.759947
13259,213,UQ02_2_2 - 0.9DL+1.2FL-EQx +0.3EQy (STAGED) - 7,1.466667,-2227.624939,36.234022,-3.317251,-0.850161,11.054547,46.317341,36406.043454,...,0.333451,0.986393,0.733380,4400.0,2.050819e+13,1.666616e+14,10454.945354,84963.021227,1.377822,0.759947
13245,213,UQ02_2_2 - 0.9DL+1.2FL-EQx +0.3EQy (STAGED) - 5,1.466667,-2227.624939,36.234022,-2.592682,-0.850161,11.054547,92.687264,36406.043454,...,-0.130149,0.986393,0.547941,4400.0,2.050819e+13,1.666616e+14,10454.945354,84963.021227,1.377822,0.567790
13247,213,UQ02_2_2 - 0.9DL+1.2FL-EQx +0.3EQy (STAGED) - 5,2.200000,-2223.248259,7.945060,-3.317251,-0.850161,10.664622,39.720663,36406.043454,...,-0.130149,0.986393,0.547941,4400.0,2.050819e+13,1.666616e+14,10454.945354,84963.021227,1.376748,0.567749
13261,213,UQ02_2_2 - 0.9DL+1.2FL-EQx +0.3EQy (STAGED) - 7,2.200000,-2223.248259,7.945060,-2.592682,-0.850161,10.664622,64.811041,36406.043454,...,0.333451,0.986393,0.733380,4400.0,2.050819e+13,1.666616e+14,10454.945354,84963.021227,1.376748,0.759893
13263,213,UQ02_2_2 - 0.9DL+1.2FL-EQx +0.3EQy (STAGED) - 7,2.933333,-2218.871579,7.945060,-2.592682,-0.850161,10.195639,36.795064,36406.043454,...,0.333451,0.986393,0.733380,4400.0,2.050819e+13,1.666616e+14,10454.945354,84963.021227,1.375676,0.759839
13249,213,UQ02_2_2 - 0.9DL+1.2FL-EQx +0.3EQy (STAGED) - 5,2.933333,-2218.871579,7.945060,-3.317251,-0.850161,10.195639,33.050704,36406.043454,...,-0.130149,0.986393,0.547941,4400.0,2.050819e+13,1.666616e+14,10454.945354,84963.021227,1.375676,0.567709
